# ArchVis RGB bounded discovery — CPU
Root가 커밋 후 source/input ZIP SHA와 새 snapshot 폴더를 고정한 ready본으로만 실행합니다.
기존 pose notebook/DB/HEAD는 변경하지 않습니다. 실제 추출·학습·전체 CRC 검증은 없습니다.
한 job: body256MiB(초기metadata 포함)/2048HTTP/600초/read512MiB/decode256MiB/RSS1GiB/local+snapshot+회수256MiB,
32member×16batch 이하. 준비 업로드/설치/auth/source 해제 공간과 시간은 phase 밖입니다.
목록 key 증가와 stereo+pose관측 교집합, 실제 추출triplet을 구분합니다. job을 재실행하지 않습니다.
실패/중단 뒤에는 마지막 회수 셀만 실행합니다. 새 job/anchor/예산은 root가 명시 인계합니다.

준비 보완: source/input을 한 개씩 선택합니다. 두 ZIP 전체 검증 뒤 새 추출 또는 정확한 기존 파일 재사용만 허용합니다.
이번 수정 notebook revision과 실제 실행 source677bc7f revision을 구분하며, 원 job/경로/예산은 root 고정값을 유지합니다.


In [ ]:
# Root가 고정할 값. 실행 source revision과 notebook 수정 revision을 구분합니다.
SOURCE_ZIP_SHA256 = ""
SOURCE_REVISION = ""  # 이번 복구본은 기존 677bc7f 실행 source 유지
NOTEBOOK_REVISION = ""  # root가 고정할 준비 notebook 수정 revision
SELECTION_SHA256 = ""  # INPUT_ROOT가 없어도 회수에 쓰는 고정 selection digest
INPUT_ZIP_SHA256 = ""
JOB_ID = "archvis_rgb_discovery_001"
SOURCE_ROOT = "/content/rewind-discovery-source"
INPUT_ROOT = "/content/rewind-discovery-inputs"
WORKSPACE = "/content/rewind-discovery/archvis_rgb_discovery_001"
SNAPSHOT_DIRECTORY = ""  # 새 별도 Drive 폴더; 기존 HEAD/pose 저장소 금지
RESUME = None  # 다음 job만: receipt_path/receipt_file_sha256/anchor를 root가 고정
INPUT_SNAPSHOT_DIRECTORY = None  # 중단 시 이전 정상 anchor 회수에 사용
RECOVERY_ZIP = "/content/archvis_rgb_discovery_001_result.zip"


In [ ]:
import hashlib, io, json, re, stat, subprocess, sys, zipfile
from pathlib import Path, PurePosixPath
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SOURCE_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", INPUT_ZIP_SHA256)
assert re.fullmatch("[a-f0-9]{64}", SELECTION_SHA256)
assert re.fullmatch("[a-f0-9]{40}", SOURCE_REVISION)
assert SNAPSHOT_DIRECTORY and not Path(WORKSPACE).exists() and not Path(WORKSPACE).is_symlink(), "소비된 job: 준비/scan 재실행 금지"
print("1/2: 고정 source ZIP을 선택하세요.")
source_uploaded = files.upload()
print("2/2: 고정 input ZIP을 선택하세요.")
input_uploaded = files.upload()
def selected(uploaded, pin, label):
    rows = [b for b in uploaded.values() if hashlib.sha256(b).hexdigest() == pin]
    assert len(rows) == 1, label + " ZIP 누락 또는 고정 SHA 불일치"
    return rows[0]
def inspect_zip(raw, maximum):
    assert len(raw) <= maximum, "압축 ZIP 크기 상한"
    payloads, directories, names = {}, set(), set()
    with zipfile.ZipFile(io.BytesIO(raw)) as z:
        rows = z.infolist()
        assert len(rows) <= 2048 and sum(i.file_size for i in rows) <= maximum, "ZIP 해제 상한"
        for item in rows:
            name = item.filename.rstrip("/")
            path = PurePosixPath(name)
            assert name and name == path.as_posix() and not path.is_absolute()
            assert ".." not in path.parts and "\\" not in name and all(":" not in part for part in path.parts), "ZIP 경로 오류"
            assert name.casefold() not in names, "ZIP 중복 경로"
            names.add(name.casefold())
            kind = stat.S_IFMT(item.external_attr >> 16)
            assert kind in (0, stat.S_IFDIR if item.is_dir() else stat.S_IFREG), "ZIP 링크/특수파일 금지"
            for parent in path.parents:
                if str(parent) != ".": directories.add(parent.as_posix())
            data = z.read(item)  # 모든 member CRC를 추출 전에 검사
            assert len(data) == item.file_size
            if item.is_dir(): directories.add(name)
            else: payloads[name] = data
    assert not set(payloads) & directories, "ZIP 파일/폴더 충돌"
    return payloads, directories
# 두 ZIP 모두 검증하기 전에는 source/input/준비 디렉터리를 만들지 않는다.
source_data = inspect_zip(selected(source_uploaded, SOURCE_ZIP_SHA256, "source"), 64*1024**2)
input_data = inspect_zip(selected(input_uploaded, INPUT_ZIP_SHA256, "input"), 8*1024**2)
assert source_data[0]["SOURCE_REVISION.txt"].decode("utf-8").strip() == SOURCE_REVISION
input_config = json.loads(input_data[0]["config.json"])
assert hashlib.sha256(json.dumps(input_config["selection"], ensure_ascii=False, sort_keys=True, separators=(",", ":"), allow_nan=False).encode()).hexdigest() == SELECTION_SHA256
def existing_matches(destination, contents):
    target = Path(destination).absolute()
    for part in (target, *target.parents):
        assert not part.is_symlink() and not getattr(part, "is_junction", lambda:False)(), "추출 경로 링크 금지"
    if not target.exists(): return False
    assert target.is_dir(), "추출 대상이 디렉터리가 아님"
    files_seen, dirs_seen = set(), set()
    for count, path in enumerate(target.rglob("*")):
        assert count < 4096 and not path.is_symlink() and not getattr(path, "is_junction", lambda:False)()
        relative = path.relative_to(target).as_posix()
        if path.is_dir(): dirs_seen.add(relative)
        else:
            assert path.is_file() and relative in contents[0], "기존 추출본의 다른 파일"
            expected = contents[0][relative]
            assert path.stat().st_size == len(expected) and path.read_bytes() == expected, "기존 추출본 바이트 불일치"
            files_seen.add(relative)
    assert files_seen == set(contents[0]) and dirs_seen == contents[1], "기존 추출본 파일/폴더 집합 불일치"
    return True
source_path, input_path = Path(SOURCE_ROOT).absolute(), Path(INPUT_ROOT).absolute()
assert not source_path.is_relative_to(input_path) and not input_path.is_relative_to(source_path)
source_exists = existing_matches(SOURCE_ROOT, source_data)
input_exists = existing_matches(INPUT_ROOT, input_data)
def materialize(destination, contents, exists):
    if exists: return  # 정확히 같은 기존 추출본만 읽기 재사용
    target = Path(destination)
    target.mkdir(parents=True, exist_ok=False)
    for name in sorted(contents[1], key=lambda s:len(PurePosixPath(s).parts)):
        (target/name).mkdir(exist_ok=True)
    for name, data in contents[0].items():
        with (target/name).open("xb") as stream: stream.write(data)
materialize(SOURCE_ROOT, source_data, source_exists)
materialize(INPUT_ROOT, input_data, input_exists)
sys.dont_write_bytecode = True
sys.path.insert(0, SOURCE_ROOT)
subprocess.check_call([sys.executable, "-m", "pip", "install", "indexed_gzip==1.10.3", "psutil"])
import indexed_gzip
assert indexed_gzip.__version__ == "1.10.3"
del source_uploaded, input_uploaded, source_data, input_data


In [ ]:
# 한 번만 실행. auth 뒤에는 600초 phase budget에 준비/scan/snapshot을 포함합니다.
from pathlib import Path
assert not Path(WORKSPACE).exists(), "이미 소비한 job: 회수 셀만 실행"
from google.colab import auth, drive
auth.authenticate_user()
drive.mount("/content/drive")
import os
mount = Path("/content/drive")
my_drive = mount/"MyDrive"
assert os.path.ismount(mount) and my_drive.is_dir(), "실제 Drive mount 확인 실패"
snapshot_path = Path(SNAPSHOT_DIRECTORY).resolve()
assert snapshot_path.is_relative_to(my_drive.resolve()) and snapshot_path != my_drive.resolve()
assert not snapshot_path.exists(), "새 별도 snapshot 폴더만 허용"
import google.auth
from google.auth.transport.requests import AuthorizedSession
from urllib3.util.retry import Retry
credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
def session_factory():
    session = AuthorizedSession(credentials)
    session._max_refresh_attempts = 0
    for adapter in session.adapters.values():
        adapter.max_retries = Retry(total=0, connect=0, read=0, redirect=0, status=0)
    return session
from rewind_discovery.runner import run
config = json.loads((Path(INPUT_ROOT)/"config.json").read_bytes())
report = run(job_id=JOB_ID, selection=config["selection"], expected_metadata=config["expected_metadata"],
    split_paths={s:Path(INPUT_ROOT)/(s+".txt") for s in ("train","validation","test")},
    split_pins=config["split_pins"], pose_catalog=Path(INPUT_ROOT)/"pose_catalog.json",
    pose_sha256=config["pose_catalog_sha256"], workspace=WORKSPACE,
    snapshot_directory=SNAPSHOT_DIRECTORY, session_factory=session_factory, resume=RESUME)
print(json.dumps({k:report.get(k) for k in ("status","body_bytes","requests","after","discovered","last_receipt")}, ensure_ascii=False))


In [ ]:
# 독립 회수: input config/인증/session/scan/run 없이 고정 selection digest로 회수합니다.
import hashlib, json, re, sys, zipfile
from pathlib import Path
from google.colab import files
assert re.fullmatch("[a-f0-9]{64}", SELECTION_SHA256)
target = Path(RECOVERY_ZIP)
if not Path(WORKSPACE).exists():
    target = target.with_name(target.stem.removesuffix("_result") + "_preparation_result" + target.suffix)
pin_path = target.with_suffix(".sha256")
def file_sha(path):
    h = hashlib.sha256()
    with path.open("rb") as stream:
        while data := stream.read(65536): h.update(data)
    return h.hexdigest()
if target.exists():
    assert pin_path.is_file() and file_sha(target) == pin_path.read_text().strip()
    with zipfile.ZipFile(target) as z:
        assert z.testzip() is None
        info = json.loads(z.read("recovery.json"))
        assert info["job_id"] == JOB_ID and info["selection_sha256"] == SELECTION_SHA256
else:
    assert not pin_path.exists()
    if Path(WORKSPACE).exists():
        # 기존 677bc7f recover와 호환. 고정 source가 존재하는 실제 job의 정상 snapshot 회수.
        assert (Path(SOURCE_ROOT)/"SOURCE_REVISION.txt").read_text().strip() == SOURCE_REVISION
        sys.dont_write_bytecode = True
        sys.path.insert(0, SOURCE_ROOT)
        from rewind_discovery.recovery import recover
        result = recover(WORKSPACE, SNAPSHOT_DIRECTORY, target, job_id=JOB_ID,
            selection_sha256=SELECTION_SHA256, input_snapshot_directory=INPUT_SNAPSHOT_DIRECTORY)
    else:
        # 준비 실패/부분 source 상태도 source 모듈 import 없이 stdlib로 회수.
        info = {"job_id":JOB_ID, "selection_sha256":SELECTION_SHA256, "last_receipt":None,
            "report_present":False, "execution_status":"workspace_absent_execution_state_unverified",
            "source_root_present":Path(SOURCE_ROOT).is_dir(), "input_root_present":Path(INPUT_ROOT).is_dir(),
            "archive_network_requested":False, "train_ready":False,
            "source_revision":SOURCE_REVISION, "notebook_revision":NOTEBOOK_REVISION}
        with zipfile.ZipFile(target, "x", zipfile.ZIP_STORED) as z:
            z.writestr("recovery.json", json.dumps(info, ensure_ascii=False, sort_keys=True))
    with pin_path.open("x") as stream: stream.write(file_sha(target)+"\n")
print("회수 ZIP SHA256:", file_sha(target))
files.download(str(target))
